# 🧠 IEEE-CIS Fraud Detection — Deep Learning ANN

> **Goal:** Train a high-performance Artificial Neural Network (ANN) using the balanced dataset.

### Pipeline:
1. Load `train_balanced.csv` and `test.csv`.
2. **Scale** features using `StandardScaler`.
3. Build a multi-layer Dense network with **Dropout** and **BatchNormalization**.
4. Evaluate using **AUC-PR** and **Confusion Matrix** on real-world imbalanced data.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Paths
DATA_DIR = os.path.abspath('../data')
TRAIN_CSV = os.path.join(DATA_DIR, 'train_balanced.csv')
TEST_CSV  = os.path.join(DATA_DIR, 'test.csv')

print("TensorFlow version:", tf.__version__)
print("Eager execution:", tf.executing_eagerly())

## 1 — Load & Scale Data

In [ ]:
print("Loading datasets...")
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y_test  = test['isFraud']

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scaling is CRITICAL for Neural Networks
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Data ready.")

## 2 — Build ANN Architecture

In [ ]:
def build_model(input_dim):
    model = models.Sequential([
        layers.Dense(256, input_dim=input_dim, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc', curve='PR')]
    )
    return model

model = build_model(X_train_scaled.shape[1])
model.summary()

## 3 — Train Model

In [ ]:
# Early stopping to prevent overfitting
early_stop = callbacks.EarlyStopping(
    monitor='val_auc', 
    patience=5, 
    restore_best_weights=True,
    mode='max'
)

print("Starting training...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=2048, # Large batch size for speed on CPU/GPU
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## 4 — Evaluation

In [ ]:
print("Evaluating on Test Set...")
y_pred_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_pred_prob)
precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall, precision)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()